# LLMOps Assignment 2 — API Cost, Latency, and Prompt Version Management
**Student Name:** Sudarsan S  
**BITS ID / Roll Number:** 2025AE05524  
**Email:** 2025AE05524@wilp.bits-pilani.ac.in  

**Course:** MLOps | **Scenario:** EduAssist AI academic Q&A assistant

Sections: **Task 1** latency · **Task 2** tokens & cost · **Task 3** MLflow Prompt Registry, evaluation, promotion.

> Runs against a **local Ollama model** (free, no API key). Start Ollama and pull the model first (see README).
> Config is read from environment variables; no secrets are hard-coded.
> `LLM_MOCK=1` runs a **dry-run with fake responses** to test the pipeline only — never submit mock results.

## 0. Configuration

In [1]:
import os, time, json, statistics, difflib, datetime, platform, sys
from datetime import timezone
import pandas as pd
import mlflow
from mlflow.genai import register_prompt, load_prompt, set_prompt_alias

# ---- Model / provider settings (edit if you use another model) ----
MODEL_NAME     = os.getenv("LLM_MODEL", "llama3.2:3b")          # free open-source model served by Ollama
API_VERSION    = "Ollama OpenAI-compatible Chat Completions API (/v1), openai-python SDK"
BASE_URL       = os.getenv("OPENAI_BASE_URL", "http://localhost:11434/v1")   # Ollama default
API_KEY        = os.getenv("OPENAI_API_KEY", "ollama")           # Ollama ignores the key; any string works
WARMUP         = os.getenv("WARMUP", "0") == "1"                 # 1 => one untimed call to load the model first

# ---- Pricing ----
# Ollama runs locally, so there is no API bill. As the assignment allows, we ESTIMATE cost using the
# published price of a hosted provider serving the same model (e.g. Llama 3.2 3B on Together/Groq/DeepInfra),
# OR your own hosting cost. LOOK UP the current price, put it below / in env vars, and record the date + source.
CURRENCY       = "USD"
INPUT_PRICE_PER_M  = float(os.getenv("INPUT_PRICE_PER_M",  "0.06"))   # per 1M input tokens  (PLACEHOLDER: verify)
OUTPUT_PRICE_PER_M = float(os.getenv("OUTPUT_PRICE_PER_M", "0.06"))   # per 1M output tokens (PLACEHOLDER: verify)
PRICING_DATE   = os.getenv("PRICING_DATE", "2026-11-03 (verify before submission)")
PRICING_SOURCE = os.getenv("PRICING_SOURCE", "<provider name + URL of the price you used>")

MAX_OUTPUT_TOKENS = 400
TEMPERATURE       = 0.0
MONTHLY_REQUESTS  = 10_000
MOCK = os.getenv("LLM_MOCK") == "1"

EXPERIMENT_NAME = "eduassist_llmops_assignment"
PROMPT_NAME     = "eduassist_qa_prompt"   # one registry prompt with versions 1, 2, 3

if MOCK:
    MODEL_NAME = "mock-model"
    print("*** MOCK MODE: results are fake and for pipeline testing only ***")
print("Python", sys.version.split()[0], "| mlflow", mlflow.__version__, "| model:", MODEL_NAME)
print(f"Estimated pricing: {INPUT_PRICE_PER_M} / {OUTPUT_PRICE_PER_M} {CURRENCY} per 1M in/out tokens, as of {PRICING_DATE}; source: {PRICING_SOURCE}")

Python 3.12.9 | mlflow 3.18.0 | model: llama3.2:3b
Estimated pricing: 0.06 / 0.06 USD per 1M in/out tokens, as of 2026-11-03 (verify before submission); source: <provider name + URL of the price you used>


## 1. Questions and prompt templates

In [2]:
QUESTIONS = {
    "Q1": "What is the difference between supervised and unsupervised learning?",
    "Q2": "Explain what overfitting is and give two ways to reduce it.",
    "Q3": "What is the time complexity of binary search and why?",
    "Q4": "What is the difference between a process and a thread in an operating system?",
    "Q5": "Briefly explain what a p-value represents in hypothesis testing.",
}

PROMPTS = {
    "v1": dict(
        commit="v1 Basic: single-line instruction, no role, no format constraints (baseline).",
        template="Answer the following student question clearly.\n\nQuestion: {{question}}",
    ),
    "v2": dict(
        commit="v2 Structured: adds academic-assistant role, concision/accuracy and undergraduate-level language.",
        template=(
            "You are an academic assistant.\n"
            "Answer the following student question in a clear, concise, and accurate manner.\n"
            "Use simple language suitable for undergraduate students.\n\n"
            "Question: {{question}}"
        ),
    ),
    "v3": dict(
        commit="v3 JSON Output: enforces strict JSON schema (answer, confidence, key_points); no markdown/code fences.",
        template=(
            "You are an academic assistant.\n"
            "Answer the student question and return ONLY valid JSON.\n"
            "Do not include markdown, explanations outside JSON, or code fences.\n\n"
            "Use this exact JSON schema:\n"
            "{\n"
            '  "answer": "string",\n'
            '  "confidence": "high | medium | low",\n'
            '  "key_points": ["string"]\n'
            "}\n\n"
            "Question: {{question}}"
        ),
    ),
}
print(PROMPTS["v3"]["template"])

You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.

Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "key_points": ["string"]
}

Question: {{question}}


## 2. LLM call helper (latency + token usage)
The endpoint, model and key come from environment variables (Ollama needs no real key). Latency uses `time.perf_counter()` (monotonic); wall-clock start/end timestamps are stored too.

In [3]:
import random

_client = None
def get_client():
    global _client
    if _client is None:
        from openai import OpenAI
        _client = OpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=300)
    return _client

def _mock_call(prompt_text):
    """Fake LLM used ONLY for pipeline testing (LLM_MOCK=1)."""
    time.sleep(random.uniform(0.05, 0.2))
    if "return ONLY valid JSON" in prompt_text:
        text = json.dumps({"answer": "mock answer", "confidence": "high", "key_points": ["a", "b"]})
    else:
        text = "Mock plain-text answer. " * 20
    pt = len(prompt_text.split()) + 5
    ct = len(text.split()) + 5
    return text, pt, ct

def check_ollama():
    """Fail early with a helpful message if Ollama / the model is not available."""
    if MOCK:
        return
    try:
        names = [m.id for m in get_client().models.list().data]
    except Exception as e:
        raise RuntimeError(f"Cannot reach Ollama at {BASE_URL}. Start it (ollama serve) -- {e!r}")
    if not any(n == MODEL_NAME or n.startswith(MODEL_NAME + ":") or n.split(":")[0] == MODEL_NAME for n in names):
        raise RuntimeError(f"Model {MODEL_NAME!r} not found. Run: ollama pull {MODEL_NAME}. Available: {names}")
    if WARMUP:
        get_client().chat.completions.create(model=MODEL_NAME, messages=[{"role": "user", "content": "hi"}], max_tokens=1)
    print("Ollama OK, model available:", MODEL_NAME, "| warm-up:", WARMUP)
check_ollama()

def call_llm(prompt_text):
    """Returns dict with response text, tokens, latency and status."""
    start_wall = datetime.datetime.now(timezone.utc)
    t0 = time.perf_counter()
    status, text, pt, ct, err = "success", "", 0, 0, ""
    try:
        if MOCK:
            text, pt, ct = _mock_call(prompt_text)
        else:
            resp = get_client().chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt_text}],
                temperature=TEMPERATURE,
                max_tokens=MAX_OUTPUT_TOKENS,
            )
            text = resp.choices[0].message.content or ""
            pt, ct = resp.usage.prompt_tokens, resp.usage.completion_tokens
    except Exception as e:
        status, err = "error", repr(e)
    latency = time.perf_counter() - t0
    end_wall = datetime.datetime.now(timezone.utc)
    return dict(response=text, input_tokens=pt, output_tokens=ct, latency_seconds=latency,
                start_time=start_wall.isoformat(), end_time=end_wall.isoformat(),
                status=status, error=err)

def cost_of(in_tok, out_tok):
    in_cost  = in_tok  / 1_000_000 * INPUT_PRICE_PER_M
    out_cost = out_tok / 1_000_000 * OUTPUT_PRICE_PER_M
    return in_cost, out_cost, in_cost + out_cost

Ollama OK, model available: llama3.2:3b | warm-up: False


## 3. Register prompt versions in MLflow Prompt Registry
We register **one prompt** (`eduassist_qa_prompt`) three times so the registry holds versions 1, 2, 3 (needed for the diff view). Version numbers map to labels v1/v2/v3.

In [4]:
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db"))
mlflow.set_experiment(EXPERIMENT_NAME)

registered = {}   # label -> registry version number
for label, p in PROMPTS.items():
    pv = register_prompt(name=PROMPT_NAME, template=p["template"], commit_message=p["commit"],
                         tags={"label": label, "model": MODEL_NAME})
    registered[label] = pv.version
    print(label, "-> registry version", pv.version)

def get_prompt_text(label, question):
    return load_prompt(f"prompts:/{PROMPT_NAME}/{registered[label]}").format(question=question)

2026/11/04 18:59:51 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/04 18:59:52 INFO mlflow.store.db.utils: Updating database tables


2026/11/04 19:00:27 INFO mlflow.tracking.fluent: Experiment with name 'eduassist_llmops_assignment' does not exist. Creating a new experiment.


v1 -> registry version 1


v2 -> registry version 2


v3 -> registry version 3


## Task 1 — Call an LLM and measure latency (2 marks)
Five academic questions, baseline prompt **v1**.

In [5]:
rows = []
for qid, q in QUESTIONS.items():
    r = call_llm(get_prompt_text("v1", q))
    rows.append(dict(question_id=qid, prompt_version="v1", model=MODEL_NAME, **r))
    print(qid, r["status"], f"{r['latency_seconds']:.3f}s", r["error"])

df1 = pd.DataFrame(rows)
latency_table = df1[["question_id","prompt_version","model","start_time","end_time","latency_seconds","status"]]
latency_table

Q1 success 90.799s 


Q2 success 82.067s 


Q3 success 58.404s 


Q4 success 84.197s 


Q5 success 18.039s 


,question_id,prompt_version,model,start_time,end_time,latency_seconds,status
0,Q1,v1,llama3.2:3b,2026-11-04T13:30:29.516356+00:00,2026-11-04T13:32:00.315258+00:00,90.798892,success
1,Q2,v1,llama3.2:3b,2026-11-04T13:32:00.318739+00:00,2026-11-04T13:33:22.385535+00:00,82.066789,success
2,Q3,v1,llama3.2:3b,2026-11-04T13:33:22.389015+00:00,2026-11-04T13:34:20.792824+00:00,58.403801,success
3,Q4,v1,llama3.2:3b,2026-11-04T13:34:20.795052+00:00,2026-11-04T13:35:44.992180+00:00,84.197122,success
4,Q5,v1,llama3.2:3b,2026-11-04T13:35:44.996342+00:00,2026-11-04T13:36:03.034859+00:00,18.038508,success


In [6]:
lat = df1.loc[df1.status == "success", "latency_seconds"]
latency_summary = pd.DataFrame({
    "metric": ["mean", "median", "min", "max", "std_dev"],
    "seconds": [lat.mean(), lat.median(), lat.min(), lat.max(), lat.std(ddof=1) if len(lat) > 1 else 0.0],
})
print(latency_summary.to_string(index=False))
slowest = df1.loc[df1.latency_seconds.idxmax()]
print("\nSlowest request:", slowest.question_id, f"{slowest.latency_seconds:.3f}s,",
      slowest.output_tokens, "output tokens")

 metric   seconds
   mean 66.701022
 median 82.066789
    min 18.038508
    max 90.798892
std_dev 29.832079

Slowest request: Q1 90.799s, 395 output tokens


**Interpretation (edit with your actual numbers after running):**
The slowest request was *<QID>* at *<X> s*. Latency of autoregressive LLMs grows with the number of **generated output tokens** (tokens are produced sequentially), so the request with the longest answer is typically slowest. Other contributors: longer prompt/context (prefill time); **cold start** (Ollama loads the model into RAM/VRAM on the first request, which can add several seconds, so the first question is often the slowest unless `WARMUP=1`); CPU vs GPU inference; other processes competing for CPU/RAM and thermal throttling on a laptop. Compare the slowest request's `output_tokens` against the others to support your explanation.

## Task 2 — Token counting and cost (3 marks)
Token counts come from `response.usage` (`prompt_tokens`, `completion_tokens`).

In [7]:
costs = df1.apply(lambda r: pd.Series(cost_of(r.input_tokens, r.output_tokens),
                  index=["input_cost","output_cost","total_cost"]), axis=1)
df2 = pd.concat([df1, costs], axis=1)
df2["total_tokens"] = df2.input_tokens + df2.output_tokens
df2["currency"] = CURRENCY
df2["pricing_date"] = PRICING_DATE
df2["api_version"] = API_VERSION

token_table = df2[["question_id","prompt_version","input_tokens","output_tokens","total_tokens",
                   "input_cost","output_cost","total_cost"]]
display(token_table.style.format({"input_cost":"{:.8f}","output_cost":"{:.8f}","total_cost":"{:.8f}"}))

avg_cost = df2.total_cost.mean()
monthly = avg_cost * MONTHLY_REQUESTS
print(f"Average cost / request : {avg_cost:.8f} {CURRENCY}")
print(f"Projected monthly cost ({MONTHLY_REQUESTS:,} requests): {monthly:.4f} {CURRENCY}")

df2.drop(columns=["response"]).to_csv("latency_tokens_cost.csv", index=False)
print("Saved latency_tokens_cost.csv")

,question_id,prompt_version,input_tokens,output_tokens,total_tokens,input_cost,output_cost,total_cost
0,Q1,v1,46,395,441,0.00000276,0.00002370,0.00002646
1,Q2,v1,48,365,413,0.00000288,0.00002190,0.00002478
2,Q3,v1,45,286,331,0.00000270,0.00001716,0.00001986
3,Q4,v1,49,400,449,0.00000294,0.00002400,0.00002694
4,Q5,v1,46,83,129,0.00000276,0.00000498,0.00000774


Average cost / request : 0.00002116 USD
Projected monthly cost (10,000 requests): 0.2116 USD


Saved latency_tokens_cost.csv


**Cost-optimization recommendation:** Even with local inference, generation dominates latency and (under hosted-equivalent pricing) cost, so the biggest lever is **capping/shortening responses** (e.g. `max_tokens`, "answer in at most 3 sentences", and the JSON schema in v3 which keeps `key_points` short). A second lever is **routing simple queries to a smaller model** (e.g. a 1B model) and **keeping the model loaded** (`OLLAMA_KEEP_ALIVE`) plus reusing the fixed instruction/schema prefix so it is not re-processed unnecessarily. Estimate the saving using your measured averages (e.g. a 30% cut in output tokens → *<X>* USD/month).

## Task 3 — Prompt versioning & MLflow evaluation (5 marks)
Run all three versions on the same five questions; log one MLflow run per version with the five required metrics.

In [8]:
REQUIRED_KEYS = {"answer", "confidence", "key_points"}
def is_valid_json_output(text):
    """Strict check: parses as JSON, exact schema keys, valid confidence, list[str] key_points."""
    try:
        obj = json.loads(text)
    except Exception:
        return False
    return (isinstance(obj, dict) and set(obj.keys()) == REQUIRED_KEYS
            and isinstance(obj["answer"], str)
            and obj["confidence"] in {"high", "medium", "low"}
            and isinstance(obj["key_points"], list)
            and all(isinstance(k, str) for k in obj["key_points"]))

all_rows, summary_rows = [], []
for label in PROMPTS:
    per_q = []
    with mlflow.start_run(run_name=f"prompt_{label}") as run:
        mlflow.log_params({"prompt_name": PROMPT_NAME, "prompt_label": label,
                           "prompt_registry_version": registered[label], "model": MODEL_NAME,
                           "temperature": TEMPERATURE, "max_tokens": MAX_OUTPUT_TOKENS,
                           "input_price_per_m": INPUT_PRICE_PER_M, "output_price_per_m": OUTPUT_PRICE_PER_M, "pricing_source": PRICING_SOURCE,
                           "currency": CURRENCY, "pricing_date": PRICING_DATE, "mock": MOCK})
        mlflow.set_tag("commit_message", PROMPTS[label]["commit"])
        for qid, q in QUESTIONS.items():
            r = call_llm(get_prompt_text(label, q))
            _, _, c = cost_of(r["input_tokens"], r["output_tokens"])
            valid = is_valid_json_output(r["response"]) if r["status"] == "success" else False
            per_q.append(dict(prompt_version=label, question_id=qid, model=MODEL_NAME, **r,
                              cost=c, valid_json=int(valid)))
        d = pd.DataFrame(per_q)
        ok = d[d.status == "success"]
        metrics = dict(
            avg_latency_seconds=ok.latency_seconds.mean(),
            avg_input_tokens=ok.input_tokens.mean(),
            avg_output_tokens=ok.output_tokens.mean(),
            avg_cost_per_request=ok.cost.mean(),
            valid_json=d.valid_json.mean(),          # valid responses / total responses
        )
        mlflow.log_metrics(metrics)
        mlflow.log_text(PROMPTS[label]["template"], "prompt_template.txt")
        mlflow.log_table(d.drop(columns=["response"]).assign(response_preview=d.response.str[:300]),
                         "per_question_results.json")
        mlflow.log_text(d.to_json(orient="records", indent=2), "raw_responses.json")
    all_rows += per_q
    summary_rows.append(dict(prompt_version=label, **metrics))
    print(label, {k: round(v, 6) for k, v in metrics.items()})

detail = pd.DataFrame(all_rows)
summary = pd.DataFrame(summary_rows)
detail.drop(columns=["response"]).assign(response_preview=detail.response.str[:300]).to_csv("prompt_evaluation_results.csv", index=False)
summary.to_csv("prompt_evaluation_summary.csv", index=False)
summary

v1 {'avg_latency_seconds': np.float64(64.393721), 'avg_input_tokens': np.float64(46.8), 'avg_output_tokens': np.float64(303.6), 'avg_cost_per_request': np.float64(2.1e-05), 'valid_json': np.float64(0.0)}


v2 {'avg_latency_seconds': np.float64(70.750754), 'avg_input_tokens': np.float64(68.8), 'avg_output_tokens': np.float64(318.4), 'avg_cost_per_request': np.float64(2.3e-05), 'valid_json': np.float64(0.0)}


v3 {'avg_latency_seconds': np.float64(20.947792), 'avg_input_tokens': np.float64(102.8), 'avg_output_tokens': np.float64(88.4), 'avg_cost_per_request': np.float64(1.1e-05), 'valid_json': np.float64(1.0)}


,prompt_version,avg_latency_seconds,avg_input_tokens,avg_output_tokens,avg_cost_per_request,valid_json
0,v1,64.393721,46.8,303.6,0.000021,0.0
1,v2,70.750754,68.8,318.4,0.000023,0.0
2,v3,20.947792,102.8,88.4,0.000011,1.0


### Prompt diff evidence (also view the registry's built-in diff in the UI)

In [9]:
def show_diff(a, b):
    ta = load_prompt(f"prompts:/{PROMPT_NAME}/{registered[a]}").template.splitlines()
    tb = load_prompt(f"prompts:/{PROMPT_NAME}/{registered[b]}").template.splitlines()
    return "\n".join(difflib.unified_diff(ta, tb, fromfile=a, tofile=b, lineterm=""))

diff_text = show_diff("v1", "v2") + "\n\n" + "="*60 + "\n\n" + show_diff("v2", "v3")
print(diff_text)
open("prompt_diff.txt", "w").write(diff_text)

--- v1
+++ v2
@@ -1,3 +1,5 @@
-Answer the following student question clearly.
+You are an academic assistant.
+Answer the following student question in a clear, concise, and accurate manner.
+Use simple language suitable for undergraduate students.
 
 Question: {{question}}


--- v2
+++ v3
@@ -1,5 +1,12 @@
 You are an academic assistant.
-Answer the following student question in a clear, concise, and accurate manner.
-Use simple language suitable for undergraduate students.
+Answer the student question and return ONLY valid JSON.
+Do not include markdown, explanations outside JSON, or code fences.
+
+Use this exact JSON schema:
+{
+  "answer": "string",
+  "confidence": "high | medium | low",
+  "key_points": ["string"]
+}
 
 Question: {{question}}


819

### Comparison table and promotion decision

In [10]:
t = summary.set_index("prompt_version")
# Rule: the deliverable requires strict JSON, so only versions with valid_json >= 0.8 are eligible;
# among eligible, pick the cheapest then fastest. If none eligible, flag for manual review.
eligible = t[t.valid_json >= 0.8]
if eligible.empty:
    best = None
    print("No version meets the JSON-validity threshold -> review v3 outputs manually!")
else:
    best = eligible.sort_values(["avg_cost_per_request", "avg_latency_seconds"]).index[0]

comparison = pd.DataFrame({
    "Prompt Version": t.index,
    "Avg Latency (s)": t.avg_latency_seconds.round(3).values,
    "Avg Input Tokens": t.avg_input_tokens.round(1).values,
    "Avg Output Tokens": t.avg_output_tokens.round(1).values,
    "Avg Cost (" + CURRENCY + ")": t.avg_cost_per_request.map(lambda x: f"{x:.8f}").values,
    "Valid JSON Rate": t.valid_json.round(2).values,
    "Recommended?": ["Yes" if v == best else "No" for v in t.index],
})
comparison.to_csv("prompt_comparison_table.csv", index=False)
comparison

,Prompt Version,Avg Latency (s),Avg Input Tokens,Avg Output Tokens,Avg Cost (USD),Valid JSON Rate,Recommended?
0,v1,64.394,46.8,303.6,0.00002102,0.0,No
1,v2,70.751,68.8,318.4,0.00002323,0.0,No
2,v3,20.948,102.8,88.4,0.00001147,1.0,Yes


In [11]:
if best:
    set_prompt_alias(PROMPT_NAME, alias="production", version=registered[best])
    prod = load_prompt(f"prompts:/{PROMPT_NAME}@production")
    print(f"Alias 'production' -> {PROMPT_NAME} version {prod.version} (label {best})")
    with mlflow.start_run(run_name="promotion_decision"):
        mlflow.set_tag("promoted_label", best)
        mlflow.log_param("production_prompt_version", prod.version)

Alias 'production' -> eduassist_qa_prompt version 3 (label v3)


### Promotion justification (>=150 words — replace the placeholders with your measured values)
I promote **v3 (JSON Output)** to `production`. *Quality:* v1 gives a bare instruction and produces free-form answers of variable length; v2 adds a role, concision and an undergraduate reading level, improving clarity, but both return unstructured prose that a downstream application cannot parse reliably. *JSON validity:* only v3 targets the required schema; it achieved a valid_json rate of **<rate>** versus **<0.00>** for v1/v2, so it is the only version that satisfies the EduAssist contract (`answer`, `confidence`, `key_points`). *Latency:* v3 averaged **<x> s** against **<y> s** (v1) and **<z> s** (v2); the structured, shorter answers usually reduce generation time, though differences across five samples are small and noisy. *Tokens:* v3 uses more input tokens because of the schema text (**<a>** vs **<b>**), but fewer/similar output tokens (**<c>**). *Cost:* average cost per request was **<$>** with a projected monthly cost of **<$>** for 10,000 requests, which is negligible versus the benefit of guaranteed machine-readable output. The trade-off is a slightly longer prompt and the need for ongoing JSON validation and guardrails (retry/repair on parse failure). Therefore v3 is recommended, with the alias allowing instant rollback to v2 if quality regresses.

## Reproducibility
Launch the UI with `mlflow ui --backend-store-uri sqlite:///mlflow.db` and take the screenshots listed in the README.